In [ ]:
#@title Step 1: Install (2 min)
!pip install gradio edge-tts gdown -q

import os
if not os.path.exists('/content/SadTalker'):
    !git clone --depth 1 https://github.com/OpenTalker/SadTalker.git /content/SadTalker

os.chdir('/content/SadTalker')
!pip install -r requirements.txt -q
!pip install dlib-bin -q
print('Install Done!')

In [ ]:
#@title Step 2: Download Models (1 min)
import os
os.chdir('/content/SadTalker')

if not os.path.exists('checkpoints/SadTalker_V0.0.2_512.safetensors'):
    !bash scripts/download_models.sh
else:
    print('Models already there!')

print('Models Ready!')

In [ ]:
#@title Step 3: Generate Video (Upload image + type script)
import gradio as gr
import subprocess, os, glob, asyncio, shutil
import edge_tts

os.chdir('/content/SadTalker')
os.makedirs('/content/output', exist_ok=True)

async def make_audio(text, voice, rate):
    audio_path = '/content/output/speech.mp3'
    comm = edge_tts.Communicate(text, voice=voice, rate=rate)
    await comm.save(audio_path)
    return audio_path

def generate(image, script, voice, rate, enhancer, still_mode, preprocess):
    if image is None:
        return None, 'Upload image first!'
    if not script.strip():
        return None, 'Type German script!'

    # Save image
    img_path = '/content/output/avatar.png'
    from PIL import Image
    Image.open(image).save(img_path)

    # TTS
    audio_path = asyncio.run(make_audio(script, voice, rate))

    # SadTalker
    cmd = [
        'python', 'inference.py',
        '--driven_audio', audio_path,
        '--source_image', img_path,
        '--result_dir', '/content/output/results',
        '--preprocess', preprocess,
        '--size', '512',
        '--expression_scale', '1.2',
    ]
    if still_mode:
        cmd.append('--still')
    if enhancer != 'none':
        cmd.extend(['--enhancer', enhancer])

    result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)

    # Find video
    vids = sorted(glob.glob('/content/output/results/**/*.mp4', recursive=True), key=os.path.getmtime, reverse=True)
    if vids:
        return vids[0], f'Done! Video: {vids[0]}'
    return None, f'Error: {result.stderr[-300:]}'

with gr.Blocks(title='Innere Staerke Video Factory') as demo:
    gr.Markdown('# Innere Staerke - Talking Avatar Generator')
    gr.Markdown('Upload avatar image, type German script, get talking video!')

    with gr.Row():
        with gr.Column():
            img_input = gr.Image(type='filepath', label='Avatar Image (PNG)')
            script_input = gr.Textbox(
                label='German Script',
                lines=8,
                value='Ich werde mit etwas beginnen, das dich vielleicht ein wenig unwohl fuehlen laesst. Wenn du merkst, dass du dich defensiv fuehlst, ist das voellig in Ordnung. Bemerke es einfach. Aber frage dich warum.'
            )
            voice_input = gr.Dropdown(
                choices=['de-DE-SeraphinaMultilingualNeural','de-DE-AmalaNeural','de-DE-KatjaNeural','de-DE-ConradNeural','de-DE-KillianNeural'],
                value='de-DE-SeraphinaMultilingualNeural',
                label='German Voice'
            )
            rate_input = gr.Dropdown(choices=['-20%','-10%','+0%','+10%','+20%'], value='+0%', label='Speech Rate')
            enhancer_input = gr.Dropdown(choices=['gfpgan','RestoreFormer','none'], value='gfpgan', label='Face Enhancer')
            still_input = gr.Checkbox(value=True, label='Still Mode (less head movement)')
            preprocess_input = gr.Dropdown(choices=['crop','resize','full'], value='full', label='Preprocess')
            btn = gr.Button('Generate Video', variant='primary')

        with gr.Column():
            video_output = gr.Video(label='Result Video')
            status = gr.Textbox(label='Status')

    btn.click(generate, [img_input, script_input, voice_input, rate_input, enhancer_input, still_input, preprocess_input], [video_output, status])

demo.launch(share=True, debug=True)